# Module 0.1: Setup & a PyTorch Crash Course

Welcome to **LLM Workout**! This is the very first hands-on notebook — the on-ramp before any math. By the end you'll be comfortable creating and manipulating **tensors**, doing matrix multiplication, moving work to a GPU, and watching PyTorch compute gradients for you. **No prior deep-learning experience is needed** — if you know basic Python, you're ready.

## 0. The Toolchain

The entire course runs on one library: **PyTorch**. You don't need to install anything by hand — from the repo root, the editable install pulls in everything (PyTorch, plotting, Jupyter, the works):

```bash
pip install -e ".[notebooks]"
```

Think of PyTorch as **NumPy with two superpowers**: it can run on a GPU, and it can compute gradients automatically. Those two superpowers are *exactly* what training a neural network needs. Everything below is a tour of the handful of operations you'll use over and over for the next 20 notebooks.

In [ ]:
import torch

# Reproducibility: fix the random seed so every run gives the SAME random numbers.
# Do this once at the top of every notebook.
torch.manual_seed(0)

print("PyTorch version:", torch.__version__)

## 1. Tensors: Containers of Numbers

### The Analogy
A **tensor** is just a container of numbers. The only thing that changes is how many directions you can travel in:

| Name | Dimensions | Everyday example |
| :--- | :--- | :--- |
| **Scalar** | 0-D | a single temperature: `37.5` |
| **Vector** | 1-D | a row of numbers: `[1, 2, 3]` |
| **Matrix** | 2-D | a spreadsheet / grid |
| **3-D tensor** | 3-D | a stack of spreadsheets (e.g. a batch of sentences) |

Every tensor has a **`.shape`** (how big it is in each direction) and a **`.dtype`** (what kind of number it holds, e.g. 32-bit float). Reading shapes is *the* survival skill in this course — when something breaks, it's almost always a shape mismatch.

In [ ]:
# Create a tensor straight from Python lists
scalar = torch.tensor(37.5)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]])

print("scalar shape:", scalar.shape, "| value:", scalar.item())
print("vector shape:", vector.shape)
print("matrix shape (rows, cols):", matrix.shape)
print("matrix dtype:", matrix.dtype)

In [ ]:
# Common factory functions: ask for a shape, get a filled tensor.
zeros = torch.zeros(2, 3)        # all zeros, shape (2, 3)
ones  = torch.ones(2, 3)         # all ones, shape (2, 3)
noise = torch.randn(2, 3)        # random values from a normal (bell-curve) distribution

print("zeros:\n", zeros)
print("ones:\n", ones)
print("randn (random, but reproducible thanks to the seed):\n", noise)

## 2. Indexing & Slicing

Tensors index just like Python lists and NumPy arrays — `[row, column]`, with `:` meaning "everything along this axis". You'll constantly grab "the last token", "the first batch", or "all features of one word" this way.

In [ ]:
m = torch.tensor([[10, 11, 12],
                  [20, 21, 22],
                  [30, 31, 32]])

print("single element [0, 2]:", m[0, 2])      # row 0, col 2 -> 12
print("whole first row  [0]   :", m[0])         # -> [10, 11, 12]
print("whole last column [:,-1]:", m[:, -1])    # every row, last col -> [12, 22, 32]
print("a 2x2 sub-block  [:2,:2]:\n", m[:2, :2])  # top-left corner

## 3. Elementwise Ops & Broadcasting

### Elementwise
Add, subtract, or multiply two tensors of the **same shape** and PyTorch just pairs up the matching positions.

### Broadcasting (the magic)
Often the shapes *don't* match — and PyTorch fixes it for you by **stretching** the smaller tensor. The rule, simply: line the shapes up from the right; a dimension of size **1** is stretched to match the other tensor.

A `(3, 1)` column plus a `(1, 4)` row "broadcasts" into a full `(3, 4)` grid — like the multiplication table you drew in school. No loops, no copies in memory.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print("elementwise add  :", a + b)      # same shape -> pairwise
print("elementwise mul  :", a * b)
print("scalar broadcast :", a * 2)      # the 2 is stretched across all 3 elements

# Now the classic broadcast: a (3,1) column + a (1,4) row -> a (3,4) grid.
col = torch.tensor([[1.0], [2.0], [3.0]])   # shape (3, 1)
row = torch.tensor([[10.0, 20.0, 30.0, 40.0]])  # shape (1, 4)
grid = col + row
print("\ncol shape", col.shape, "+ row shape", row.shape, "-> grid shape", grid.shape)
print(grid)

## 4. Matrix Multiplication — THE Operation

This is the single most important operation in the whole course. A Transformer is, at heart, a tall stack of matrix multiplications. PyTorch writes it with the `@` operator (or `torch.matmul`).

### The shape rule (memorize this!)
$$(a, b) \;@\; (b, c) \;=\; (a, c)$$

The **inner** dimensions must match (`b == b`); they cancel out. The **outer** dimensions (`a` and `c`) become the result's shape. If the inner dimensions don't match, PyTorch raises an error — and that's the #1 bug you'll hit, so always check shapes.

In [ ]:
A = torch.randn(2, 3)   # (a=2, b=3)
B = torch.randn(3, 4)   # (b=3, c=4)

C = A @ B               # same as torch.matmul(A, B)
print("(2,3) @ (3,4) -> ", C.shape)   # inner 3s cancel -> (2, 4)

# Proof the two spellings are identical:
print("@ and torch.matmul agree:", torch.allclose(A @ B, torch.matmul(A, B)))

## 5. Reshaping & Transposing

The numbers in a tensor stay the same, but we constantly need to **re-organize** them into a different shape. Two tools do almost all the work:

- **`.view` / `.reshape`** — keep the same data, lay it out in a new shape (the total number of elements must stay equal). Pass `-1` to let PyTorch infer one dimension for you.
- **`.transpose(dim0, dim1)`** — swap two axes (rows ↔ columns for a 2-D tensor). You'll use this *constantly* in the attention notebooks to line up Queries and Keys.

In [ ]:
t = torch.arange(12)            # a flat vector: 0,1,...,11  (shape (12,))
print("original:", t.shape)

as_matrix = t.view(3, 4)        # reinterpret the 12 numbers as a 3x4 grid
print("viewed as (3,4):\n", as_matrix)

auto = t.reshape(2, -1)         # -1 means "you figure out this dimension" -> (2, 6)
print("reshaped to (2,-1) ->", auto.shape)

transposed = as_matrix.transpose(0, 1)   # swap rows and cols -> (4, 3)
print("transposed (3,4)->(4,3):\n", transposed)

## 6. Batched Matrix Multiplication (the shape Transformers really use)

So far every matrix has been 2-D. But real Transformer data has **more axes**. A
batch of sentences is a **3-D** tensor:

$$(\text{batch},\ \text{sequence length},\ \text{features})$$

and once attention splits things into multiple "heads" it becomes **4-D**:

$$(\text{batch},\ \text{heads},\ \text{sequence length},\ \text{features})$$

The good news: **`@` still works, unchanged.** When a tensor has more than two
axes, `torch.matmul` treats the **last two** axes as the matrix to multiply, and
every axis *before* that as a **batch dimension it loops over for you, in
parallel**:

$$(B,\ H,\ T,\ d)\ @\ (B,\ H,\ d,\ T)\ =\ (B,\ H,\ T,\ T)$$

Only the last two dims (`T,d` and `d,T`) do the real matmul — the inner `d`s cancel
just like in section 4. The leading `B` and `H` come along for the ride: the same
2-D multiply is done independently for every sentence and every head. This *one*
line is how attention scores every token against every other token, for every
sentence and head at once — so let's make the shapes concrete.

In [ ]:
# A batch of 2 items, each a (4 x 3) matrix, times a batch of 2 (3 x 5) matrices.
A = torch.randn(2, 4, 3)     # (batch=2, rows=4, inner=3)
B = torch.randn(2, 3, 5)     # (batch=2, inner=3, cols=5)
C = A @ B                     # matmul over the LAST two dims, batched over the first
print("(2,4,3) @ (2,3,5) ->", tuple(C.shape), "  (the leading 2 just means 'do it twice')")

# Proof it equals looping the plain 2-D matmul over the batch by hand:
manual = torch.stack([A[0] @ B[0], A[1] @ B[1]])
print("batched == looped by hand:", torch.allclose(C, manual))

# The exact shape attention uses: (batch, heads, seq, head_dim).
Q = torch.randn(2, 8, 10, 16)          # 2 sentences, 8 heads, 10 tokens, 16 features
K = torch.randn(2, 8, 10, 16)
scores = Q @ K.transpose(-2, -1)       # (...,10,16) @ (...,16,10) -> (...,10,10)
print("\nQ", tuple(Q.shape), "@ Kᵀ ->", tuple(scores.shape))
print("Read it as: (batch=2, heads=8, 10 queries x 10 keys) -- every token vs every token.")

> **How to read a 4-D shape.** Walk the axes left to right and name each one out
> loud: `(2, 8, 10, 16)` → *"2 sentences, 8 heads each, 10 tokens each, 16 numbers
> per token."* When a later notebook prints a shape, do exactly this — naming the
> axes is the fastest way to understand (and debug) any Transformer. The full story
> of *why* there are heads comes in the attention notebook; here you only need to
> know the shape is legal and `@` handles it.

## 7. Devices: CPU vs GPU

A tensor lives on a **device**. By default that's the **CPU**. But a neural network does *billions* of those matrix multiplications, and a **GPU** can do thousands of them in parallel — often 10–100× faster. (Apple Silicon Macs expose their GPU as **MPS**; NVIDIA cards as **CUDA**.)

The pattern you'll see everywhere: detect the best available device once, then `.to(device)` your tensors and models. If no GPU is present, everything still runs on CPU — just slower.

In [ ]:
# Pick the best device available: CUDA (NVIDIA) > MPS (Apple) > CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Move a tensor onto that device. The math is identical; only the hardware changes.
x = torch.randn(2, 2).to(device)
print("x lives on:", x.device)
print("x @ x still works:\n", x @ x)

## 8. Autograd — Why PyTorch Can Learn

Here's the second superpower, and the reason any of this trains at all.

When you mark a tensor with **`requires_grad=True`**, PyTorch quietly **records every operation** you do to it, building a little graph behind the scenes. Then, when you call **`.backward()`** on a result, it walks that graph in reverse and computes the **gradient** — how much each input would need to change to nudge the output. The gradient lands in the input's **`.grad`** attribute.

**Why this matters:** training a model = "which way should I tweak each parameter to reduce the error?" That question *is* a gradient. PyTorch computes it for us automatically, so we never derive calculus by hand.

Let's verify it on something we can check by hand. For $y = x^2$, calculus says the derivative is $\frac{dy}{dx} = 2x$. So at $x = 3$, the gradient should be $2 \times 3 = 6$.

In [ ]:
# 1. Make a tensor we want gradients for.
x = torch.tensor(3.0, requires_grad=True)

# 2. Do some math. PyTorch is silently recording these steps.
y = x ** 2

# 3. Ask: how does y change as x changes? (run backprop)
y.backward()

# 4. Read the answer off x.grad.
print("x      =", x.item())
print("y = x^2 =", y.item())
print("dy/dx (PyTorch) =", x.grad.item(), "  <- should be 2*x = 6")

That `6` came out of thin air — we never wrote the derivative ourselves. Multiply this over millions of parameters and you have the engine of deep learning.

This is exactly the machinery the **training loop** will lean on. We unpack *why* gradients point "downhill" toward a better model in **Module 0.2** (probability & calculus), and we put it to work for real in the **training loop notebook**. For now, the takeaway is simply: *PyTorch remembers operations and computes gradients for us.*

## 9. Five PyTorch Idioms You'll See Everywhere

These five appear constantly from here on. None is hard, but all are easy to be
confused by if nobody tells you what they do — so here they are, once, up front.

**1. `.unsqueeze(dim)` — add a size-1 dimension.** Models expect a *batch* dimension
even when you have one item. `unsqueeze` inserts an axis of length 1 at the position
you name; `.squeeze()` removes size-1 axes again. Shape bookkeeping, nothing more.

**2. `with torch.no_grad():` — "I'm just looking."** Normally PyTorch records every
operation so it can compute gradients later (Section 8). When you're only *inspecting*
or *generating* — not training — that recording is wasted memory and time. This block
switches it off.

**3. `.detach()` — cut one tensor loose from the graph.** Same idea, but for a single
tensor: it returns a copy that carries the numbers but no gradient history. You'll see
it when printing or plotting a value that came out of a model.

**4. `model.train()` / `model.eval()` — a *mode switch*, not an action.** Neither one
trains or evaluates anything! They just flip a flag that tells certain layers (like
dropout) to behave differently during training vs. inference. Forgetting `model.eval()`
before generating is a classic bug, so it's worth knowing early.

**5. `register_buffer(...)` — a tensor that travels with the model but never learns.**
Parameters get gradients and get updated; a *buffer* is saved and moved to the GPU
alongside them but is deliberately **not** trained. We use it for fixed lookup tables
like positional encodings.

In [ ]:
import torch
import torch.nn as nn

# 1. unsqueeze: add a dimension -------------------------------------------------
v = torch.tensor([1.0, 2.0, 3.0])
print("original         ", tuple(v.shape))
print("unsqueeze(0)     ", tuple(v.unsqueeze(0).shape), " <- became 1 row  (a batch of 1)")
print("unsqueeze(1)     ", tuple(v.unsqueeze(1).shape), " <- became 1 column")
print("squeeze() undoes ", tuple(v.unsqueeze(0).squeeze().shape))

# 2 & 3. no_grad and detach -----------------------------------------------------
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
print("\ntracked tensor        :", y.requires_grad, " (PyTorch is recording)")
print("after .detach()       :", y.detach().requires_grad, " (numbers kept, history dropped)")
with torch.no_grad():
    z = x ** 2
print("inside no_grad block  :", z.requires_grad, " (nothing recorded at all)")

# 4. train / eval mode ----------------------------------------------------------
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(p=0.5))
model.train(); print("\nmodel.training after .train():", model.training)
model.eval();  print("model.training after .eval() :", model.training, " <- dropout now disabled")

# 5. register_buffer: saved, moved, but never trained ---------------------------
class Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(3))   # LEARNS
        self.register_buffer("lookup", torch.arange(3.0))  # travels along, never learns

d = Demo()
print("\nparameters (these get gradients):", [n for n, _ in d.named_parameters()])
print("buffers    (these do not)       :", [n for n, _ in d.named_buffers()])

### 🏋️ Try it yourself

1. **Shape detective.** Create `P = torch.randn(4, 5)` and `Q = torch.randn(5, 2)`. Before running it, predict the shape of `P @ Q`, then print `(P @ Q).shape` to check. Bonus: what error do you get if you try `Q @ P`, and why?
2. **Gradient by hand.** Set `w = torch.tensor(2.0, requires_grad=True)`, compute `z = 3 * w ** 2`, call `z.backward()`, and print `w.grad`. Work out `dz/dw` with pencil and paper (it's `6 * w`) and confirm PyTorch agrees.

In [ ]:
import torch

# --- Exercise 1: shape detective ---
P = torch.randn(4, 5)
Q = torch.randn(5, 2)
# TODO: predict the shape, then print (P @ Q).shape
# TODO (bonus): try Q @ P and read the error message


# --- Exercise 2: gradient by hand ---
w = torch.tensor(2.0, requires_grad=True)
# TODO: z = 3 * w ** 2
# TODO: z.backward()
# TODO: print(w.grad)   # should equal 6 * w = 12